# Lezione 10 — La prima rete neurale

**Come si usa questo notebook.** Come sempre. Prerequisiti: Lezioni 7-9 - il prodotto scalare come somma pesata (Lezione 7), la discesa del gradiente (Lezione 8), softmax e cross-entropy (Lezione 9). Da qui comincia la
**Fase 2**: TensorFlow/Keras entra nello stack — e scoprirai che sai già
quasi tutto quello che fa.

**Cosa saprai fare alla fine:** costruire, addestrare e valutare una rete
neurale con Keras, e — più importante — sapere *cosa* c'è dentro ogni riga,
perché ogni pezzo l'hai già scritto a mano nella Fase 0.

## Parte 1 — Teoria: cosa aggiunge una rete al modello lineare

La softmax regression della Lezione 9 (i punteggi `X @ W + b` trasformati in probabilità con la softmax, addestrati minimizzando la cross-entropy) è `softmax(X @ W + b)`: **un solo strato**. La sua forza è anche il suo limite: può tracciare solo confini
**lineari** tra le classi. Se le classi si separano con una linea (o un
piano), basta; se il confine è curvo, non c'è training che tenga.

Una rete neurale impila strati:

```text
h = attivazione(X @ W1 + b1)      <- strato nascosto
y = softmax(h @ W2 + b2)          <- strato di uscita
```

Ogni strato è la somma pesata della Lezione 7 (il prodotto scalare `x · w`, esteso a più neuroni). La novità decisiva è
l'**attivazione non lineare** tra gli strati — la più usata è la **ReLU**:
`max(0, x)`, tiene i positivi e azzera i negativi. Sembra poco, ma è tutto:

> **senza non-linearità, impilare strati è inutile**: la composizione di
> funzioni lineari è ancora lineare (`W2 @ (W1 @ x) = (W2 W1) @ x` — un
> solo strato mascherato). Con la ReLU in mezzo, ogni neurone nascosto
> "piega" lo spazio, e la rete può disegnare confini curvi arbitrari.

Il resto è nomenclatura: uno **strato denso** (Dense) è `X @ W + b` con
attivazione; i **neuroni** sono le colonne di W (ogni neurone = una somma pesata); il training è la discesa del gradiente (Lezione 8) sulla cross-entropy (Lezione 9), con i gradienti calcolati automaticamente invece che a mano - l'autodiff, che apriremo nella Lezione 11.

## Parte 2 — Vederlo: quando il lineare non basta

Il modo più onesto per capire il valore della non-linearità è un dataset in
cui il confine giusto è **curvo**: due mezzelune intrecciate. Un modello
lineare non può separarle per costruzione; una piccola rete sì. Guarda:

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # silenzia i log informativi di TF

import numpy as np
import pandas as pd
import keras

keras.utils.set_random_seed(42)            # riproducibilita' (Lezione 3!)
print('Keras', keras.__version__)

In [ ]:
from sklearn.datasets import make_moons

X_lune, y_lune = make_moons(n_samples=300, noise=0.2, random_state=0)

lineare = keras.Sequential([keras.layers.Dense(1, activation='sigmoid')])
lineare.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lineare.fit(X_lune, y_lune, epochs=200, verbose=0)

rete = keras.Sequential([
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])
rete.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
rete.fit(X_lune, y_lune, epochs=200, verbose=0)

print(f"lineare  : {lineare.evaluate(X_lune, y_lune, verbose=0)[1]:.0%}")
print(f"con ReLU : {rete.evaluate(X_lune, y_lune, verbose=0)[1]:.0%}")

In [ ]:
import matplotlib.pyplot as plt

xx, yy = np.meshgrid(np.linspace(-2, 3, 150), np.linspace(-1.5, 2, 150))
griglia = np.c_[xx.ravel(), yy.ravel()]

fig, assi = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for asse, modello, titolo in [(assi[0], lineare, 'Modello lineare'),
                              (assi[1], rete, 'Rete con strato ReLU')]:
    Z = modello.predict(griglia, verbose=0).reshape(xx.shape)
    asse.contourf(xx, yy, Z, levels=[0, 0.5, 1], alpha=0.25)
    asse.scatter(X_lune[:, 0], X_lune[:, 1], c=y_lune, s=8)
    asse.set_title(titolo)
plt.tight_layout()
plt.show()

A sinistra il confine è per forza una retta, e taglia male le mezzelune; a
destra lo strato nascosto lo ha **piegato** attorno ai dati. Questo grafico
è la ragione d'essere del deep learning in un'immagine.

## Parte 3 — Teoria: le tre righe di Keras, tradotte

```python
model = keras.Sequential([...])    # la catena di strati (l'architettura)
model.compile(loss=..., optimizer=..., metrics=[...])
model.fit(X, y, epochs=..., validation_data=...)
```

- `Sequential` — la composizione di funzioni: strato dopo strato, come la chain rule della Lezione 8 applica le derivate a catena;
- `loss='sparse_categorical_crossentropy'` — esattamente la cross-entropy della Lezione 9 ("sparse" = il target è l'intero della classe, come nel nostro CSV, non il one-hot);
- `optimizer='adam'` — una discesa del gradiente più furba di quella a passo fisso della Lezione 8 (adatta il passo per parametro); il learning rate resta la manopola che conosci;
- `metrics=['accuracy']` — la metrica per te; la loss resta il contratto per il modello, e possono raccontare storie diverse (lo vedrai bene nella Lezione 13);
- `fit` — il ciclo previsione → errore → gradiente → passo che hai scritto a mano nella Lezione 9, con l'aggiunta dei batch (piccoli gruppi di esempi alla volta, invece di tutto il dataset insieme), spiegati in dettaglio nella Lezione 11.

Nessuna magia nuova: solo i tuoi concetti, automatizzati.

## Parte 4 — Esercizio guidato

Nella cella sotto, costruisci **tu** la rete per le mezzelune, variando una
sola cosa: lo strato nascosto con **2 neuroni** invece di 16. Addestra 200
epoche e confronta l'accuratezza con i due modelli della Parte 2. Prima di
eseguire: scommetti dove si piazzerà, e perché?

**Prova tu:**

In [ ]:
keras.utils.set_random_seed(1)

# Scrivi qui: Sequential con Dense(2, relu) + Dense(1, sigmoid),
# compile come sopra, fit 200 epoche su X_lune / y_lune, poi evaluate.

pass

### Soluzione spiegata

Con 2 neuroni la rete può fare solo due "pieghe": di solito finisce tra il
lineare e la rete da 16 — meglio della retta, peggio del confine morbido.
La **capacità** (quanti neuroni/strati) determina quanto la rete può piegare lo spazio: poca = non basta, troppa = memorizza i dati invece di imparare la regola generale sottostante (un fenomeno che hai già incontrato nella Lezione 3 con lo studente che studia le soluzioni a memoria: qui si chiama overfitting, e la Lezione 12 lo mostra dal vivo su questa stessa rete).

In [ ]:
keras.utils.set_random_seed(1)
piccola = keras.Sequential([
    keras.layers.Dense(2, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])
piccola.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
piccola.fit(X_lune, y_lune, epochs=200, verbose=0)
print(f"2 neuroni nascosti: {piccola.evaluate(X_lune, y_lune, verbose=0)[1]:.0%}")

## Parte 5 — Il progetto: Memory AI Lab, passo 10

Il momento che il corso prepara da nove lezioni: la prima rete neurale del progetto affronta l'asticella salvata nella Lezione 9 (i pesi della softmax regression fatta a mano, salvati su disco). Stesse feature, stessi split onesti, stesso protocollo: si confronta su **validation** (il test resta sigillato fino alla valutazione finale, regola della Lezione 3).

In [ ]:
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
insiemi = {n: pd.read_csv(processed / f'memory_features_{n}.csv') for n in ['train', 'val']}
X = {n: f.drop(columns='target').to_numpy().astype('float32') for n, f in insiemi.items()}
y = {n: f['target'].to_numpy() for n, f in insiemi.items()}

# l'asticella della Fase 0
W0, b0 = np.load(processed / 'softmax_baseline_W.npy'), np.load(processed / 'softmax_baseline_b.npy')
baseline_val = ((X['val'] @ W0 + b0).argmax(axis=1) == y['val']).mean()

keras.utils.set_random_seed(42)
modello = keras.Sequential([
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(4, activation='softmax'),
])
modello.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
                metrics=['accuracy'])
storia = modello.fit(X['train'], y['train'], epochs=150,
                     validation_data=(X['val'], y['val']), verbose=0)

acc_val = modello.evaluate(X['val'], y['val'], verbose=0)[1]
print(f'softmax a mano (Lezione 9): {baseline_val:.0%} su validation')
print(f'rete Keras (16 neuroni)   : {acc_val:.0%} su validation')

Leggi il confronto con l'onestà delle Lezioni 3-4 (confronto solo su validation, mai sul test consumato): se la rete supera la baseline, lo strato nascosto ha trovato combinazioni di feature che il
modello lineare non vede. Se il vantaggio è piccolo o nullo, la diagnosi è
un'altra — e più preziosa: **il collo di bottiglia sono le feature**
(lunghezze e flag dicono poco sul *significato* di una memoria), non il
modello. È la motivazione esatta della Fase 3: gli embedding del testo.

## Cosa hai imparato

- Un modello lineare traccia solo confini lineari; l'**attivazione non
  lineare** (ReLU) tra gli strati è ciò che permette confini curvi.
- Senza non-linearità, impilare strati collassa in un solo strato.
- `Sequential` / `compile` / `fit` = architettura / contratto (loss + optimizer) / il ciclo di training a mano della Lezione 9, automatizzato.
- La **capacità** della rete si sceglie, e ha un prezzo: la Lezione 12 mostra cosa succede quando è troppa.
- I confronti si fanno su validation, contro una baseline dichiarata.

## Quiz

1. Perché due strati Dense *senza* attivazione in mezzo equivalgono a uno
   solo?
2. Cosa significa "sparse" in `sparse_categorical_crossentropy`?
3. La rete batte la baseline sul train ma non su validation: tra le ipotesi
   possibili, quale controlli per prima e con quale strumento già visto
   nel corso (pensa a come si confrontano due curve nel tempo)?

<details>
<summary><b>Apri le risposte</b></summary>

1. La composizione di funzioni lineari è lineare: `W2 @ (W1 @ x + b1) + b2`
   si riscrive come un'unica moltiplicazione `W @ x + b`. Serve la
   non-linearità per aggiungere potere espressivo.
2. Che il target è l'indice intero della classe (0..3) invece del vettore
   one-hot; la loss è identica, cambia solo il formato dell'etichetta.
3. Overfitting: la rete memorizza il train invece di imparare la regola generale (lo stesso fenomeno dello studente della Lezione 3). Si controlla confrontando le curve di loss train/validation lungo le epoche - la Lezione 12 lo mostra dal vivo; lo strumento è la `storia` che `fit` restituisce.
</details>

## Fonti

- Keras, *The Sequential model*: https://keras.io/guides/sequential_model/
- Keras, `Dense` layer: https://keras.io/api/layers/core_layers/dense/
- TensorFlow, *Basic classification tutorial*:
  https://www.tensorflow.org/tutorials/keras/classification

## Prossima lezione

`fit` ha fatto tutto da solo, nascondendo dentro di sé i batch e l'autodiff. Nella Lezione 11 apriamo la scatola: cosa sono i batch, come funziona l'autodiff che calcola i gradienti al posto tuo, e riscriviamo il loop di training a mano, in TensorFlow, per vedere che produce lo stesso identico risultato.